# PPO Training Notebook for OfficeAgentEnv
Using TRL + LLM policy training


In [ ]:
!pip install -q trl transformers accelerate bitsandbytes

In [ ]:
import json
import random
import re
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

from env.environment import ExecAssistEnv
from env.models import ActionType, EmailCategory, ExecAssistAction


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)


In [ ]:
MAX_STEPS = {"easy": 10, "medium": 15, "hard": 12}
TRAIN_TASKS = ["easy", "medium", "hard"]


def format_observation_as_text(obs: Dict[str, Any]) -> str:
    pending = obs.get("pending_emails", [])
    calendar = obs.get("calendar_events", [])
    last_result = obs.get("last_action_result", "")

    pending_text = []
    for email in pending[:6]:
        pending_text.append(
            f"id={email['email_id']} from={email['sender']} subject={email['subject']} body={email['body'][:180]}"
        )

    calendar_text = [
        f"{ev['start_time']} -> {ev['end_time']} ({ev['title']})"
        for ev in calendar[:6]
    ]

    return (
        f"Task: {obs.get('task_name', '')}\n"
        f"Step: {obs.get('current_step', 0)}\n"
        f"Last result: {last_result}\n"
        f"Pending emails ({len(pending)}):\n- "
        + "\n- ".join(pending_text if pending_text else ["none"])
        + "\nCalendar:\n- "
        + "\n- ".join(calendar_text if calendar_text else ["none"])
        + "\nReturn ONE JSON action with keys matching ExecAssistAction."
    )


def _safe_action_json(text: str) -> Dict[str, Any]:
    cleaned = text.replace("```json", "").replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output")
    return json.loads(cleaned[start : end + 1])


def parse_action(action_text: str, obs: Dict[str, Any]) -> Dict[str, Any]:
    parsed = _safe_action_json(action_text)
    pending = obs.get("pending_emails", [])
    valid_ids = {e["email_id"] for e in pending}

    if "action_type" not in parsed or parsed["action_type"] not in {
        ActionType.CLASSIFY_EMAIL.value,
        ActionType.REPLY_EMAIL.value,
        ActionType.SCHEDULE_MEETING.value,
        ActionType.IGNORE_EMAIL.value,
    }:
        raise ValueError("Invalid or missing action_type")

    if "email_id" not in parsed or parsed["email_id"] not in valid_ids:
        raise ValueError("Invalid or missing email_id")

    if parsed["action_type"] == ActionType.CLASSIFY_EMAIL.value:
        if parsed.get("category") not in {c.value for c in EmailCategory}:
            raise ValueError("Invalid category for classify_email")

    if parsed["action_type"] == ActionType.REPLY_EMAIL.value:
        parsed.setdefault("reply_text", "Thanks for the message. I will follow up shortly.")

    if parsed["action_type"] == ActionType.SCHEDULE_MEETING.value:
        parsed.setdefault("meeting_title", "Meeting")
        parsed.setdefault("meeting_start_time", "2024-07-01 11:00")
        parsed.setdefault("meeting_end_time", "2024-07-01 11:30")
        if not parsed.get("participants"):
            sender = next(e["sender"] for e in pending if e["email_id"] == parsed["email_id"])
            parsed["participants"] = [sender]

    return parsed


In [ ]:
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=2,
    mini_batch_size=1,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
)


def model_generate(obs_text: str) -> str:
    inputs = tokenizer(obs_text, return_tensors="pt", truncation=True, max_length=1024).to(model.pretrained_model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
def run_episode(task: str, seed: int, update_policy: bool) -> tuple[float, List[Dict[str, Any]]]:
    env = ExecAssistEnv(task_name=task, seed=seed)
    obs = env.reset().model_dump()
    done = False
    total_reward = 0.0
    trajectory: List[Dict[str, Any]] = []

    while not done:
        obs_text = format_observation_as_text(obs)
        action_text = model_generate(obs_text)
        action = parse_action(action_text, obs)

        step_result = env.step(ExecAssistAction(**action)).model_dump()
        reward = float(step_result["reward"])
        done = bool(step_result["done"])

        if update_policy:
            ppo_trainer.step([obs_text], [action_text], [reward])

        trajectory.append(
            {
                "observation": obs_text,
                "action": action_text,
                "reward": reward,
            }
        )

        total_reward += reward
        obs = step_result["observation"]

        print(f"[STEP] task={task} action={json.dumps(action)} reward={reward:.4f}")

    print(f"[EPISODE] task={task} seed={seed} total_reward={total_reward:.4f}")
    return total_reward, trajectory


def evaluate(num_episodes: int = 6) -> float:
    rewards = []
    for idx in range(num_episodes):
        task = random.choice(TRAIN_TASKS)
        seed = random.randint(1, 10_000_000)
        reward, _ = run_episode(task=task, seed=seed, update_policy=False)
        rewards.append(reward)
    return sum(rewards) / max(1, len(rewards))


print("[BASELINE] Evaluating before PPO training...")
before_avg_reward = evaluate(num_episodes=4)

TRAIN_EPISODES = 24
reward_history: List[float] = []
trajectories: List[Dict[str, Any]] = []

print("[TRAIN] Running PPO training loop...")
for episode in range(1, TRAIN_EPISODES + 1):
    task = random.choice(TRAIN_TASKS)
    seed = random.randint(1, 10_000_000)
    total_reward, trajectory = run_episode(task=task, seed=seed, update_policy=True)
    reward_history.append(total_reward)
    trajectories.append(
        {
            "episode": episode,
            "task": task,
            "seed": seed,
            "total_reward": total_reward,
            "steps": trajectory,
        }
    )

print("[EVAL] Evaluating after PPO training...")
after_avg_reward = evaluate(num_episodes=4)

print(f"Before: {before_avg_reward:.4f}")
print(f"After: {after_avg_reward:.4f}")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(reward_history) + 1), reward_history, marker="o", linewidth=1.2)
plt.title("PPO Training Reward Curve")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(alpha=0.3)
plt.show()
